In [1]:
import os
import cv2
import numpy as np
import onnxruntime as ort
from PIL import Image
from datetime import datetime
from gi.repository import Gtk, GdkPixbuf, GLib
from RPLCD.i2c import CharLCD

# === LCD SETUP ===
lcd = CharLCD(i2c_expander='PCF8574', address=0x27, port=1,
              cols=16, rows=2, charmap='A02', auto_linebreaks=True)

# === MODEL SETUP ===
onnx_model_path = '/home/piji/500mobilenet_tomato_disease.onnx'
session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
target_size = (224, 224)

# === CLASS NAMES ===
class_names = [
    "Tomato Bacterial Spot", "Tomato Early Blight","Tomato Healthy", "Tomato Late Blight",
    "Tomato Leaf Mold","Tomato Mosaic Virus", "Tomato Septoria Leaf Spot", "Tomato Spider Mites (Two-Spotted Spider Mite)",
    "Tomato Target Spot", "Tomato Yellow Leaf Curl Virus"
]

# === LOAD IMAGES ===
image_dir = '/home/piji/tomato_disease_gui/datasetsplit4080/test'
image_label_pairs = []

for label_folder in os.listdir(image_dir):
    class_path = os.path.join(image_dir, label_folder)
    if not os.path.isdir(class_path):
        continue
    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
            img_path = os.path.join(class_path, filename)
            image_label_pairs.append((img_path, label_folder))

image_label_pairs.sort()
image_index = 0


def preprocess_image(img_path):
    image = Image.open(img_path).convert('RGB')
    image = image.resize(target_size)
    img_array = np.array(image).astype(np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def predict(img_array):
    output = session.run(None, {input_name: img_array})[0]
    pred_index = np.argmax(output)
    confidence = np.max(output)
    return class_names[pred_index], confidence


class DetectionGUI:
    def __init__(self):
        self.builder = Gtk.Builder()
        self.window = Gtk.Window(title="Tomato Leaf Test Detection")
        self.window.set_default_size(750, 600)
        self.window.connect("destroy", Gtk.main_quit)

        vbox = Gtk.Box(orientation=Gtk.Orientation.VERTICAL, spacing=6)
        self.image = Gtk.Image()
        self.label = Gtk.Label(label="Prediction and Ground Truth will appear here.")

        # === Tombol ===
        self.button_box = Gtk.Box(spacing=10)
        self.prev_button = Gtk.Button(label="Previous")
        self.prev_button.connect("clicked", self.on_prev_image)

        self.next_button = Gtk.Button(label="Next")
        self.next_button.connect("clicked", self.on_next_image)

        self.screenshot_button = Gtk.Button(label="Screenshot")
        self.screenshot_button.connect("clicked", self.on_screenshot)

        self.record_button = Gtk.Button(label="Start Recording")
        self.record_button.connect("clicked", self.toggle_recording)

        self.button_box.pack_start(self.prev_button, True, True, 0)
        self.button_box.pack_start(self.next_button, True, True, 0)
        self.button_box.pack_start(self.screenshot_button, True, True, 0)
        self.button_box.pack_start(self.record_button, True, True, 0)

        vbox.pack_start(self.image, True, True, 0)
        vbox.pack_start(self.label, False, False, 0)
        vbox.pack_start(self.button_box, False, False, 0)

        self.window.add(vbox)
        self.window.show_all()

        self.recording = False
        self.video_writer = None

        self.on_next_image(None)

    def on_prev_image(self, _):
        global image_index
        if image_index > 1:
            image_index -= 2
        else:
            image_index = 0
        self.on_next_image(None)

    def on_next_image(self, _):
        global image_index
        if image_index >= len(image_label_pairs):
            self.label.set_text("All images have been processed.")
            lcd.clear()
            lcd.write_string("All done")
            return

        img_path, true_label = image_label_pairs[image_index]
        image_index += 1

        img_array = preprocess_image(img_path)
        pred_label, confidence = predict(img_array)

        is_correct = (true_label.lower() == pred_label.lower())
        status = "✅" if is_correct else "❌"

        text = f"{status} GT: {true_label} | Pred: {pred_label} ({confidence*100:.2f}%)"
        self.label.set_text(text)
        self.update_image(img_path)

        # === LCD ===
        lcd.clear()
        lcd.write_string(pred_label[:16])
        lcd.cursor_pos = (1, 0)
        lcd.write_string(f"{confidence*100:.1f}%")

        # === Rekam ke video jika aktif ===
        if self.recording and self.video_writer:
            frame = cv2.imread(img_path)
            if frame is not None:
                frame = cv2.resize(frame, (640, 480))
                cv2.putText(frame, f"{pred_label} {confidence*100:.1f}%", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                self.video_writer.write(frame)

    def update_image(self, img_path):
        pixbuf = GdkPixbuf.Pixbuf.new_from_file_at_scale(img_path, 640, 480, True)
        self.image.set_from_pixbuf(pixbuf)
        self.last_image_path = img_path  # For screenshot use

    def on_screenshot(self, _):
        now = datetime.now().strftime('%Y%m%d_%H%M%S')
        screenshot_path = f"screenshot_{now}.png"
        img = cv2.imread(self.last_image_path)
        if img is not None:
            cv2.imwrite(screenshot_path, img)
            print(f"[INFO] Screenshot saved: {screenshot_path}")

    def toggle_recording(self, _):
        if not self.recording:
            now = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"recorded_{now}.avi"
            self.video_writer = cv2.VideoWriter(filename,
                                                cv2.VideoWriter_fourcc(*'XVID'),
                                                5, (640, 480))
            self.recording = True
            self.record_button.set_label("Stop Recording")
            print(f"[INFO] Recording started: {filename}")
        else:
            self.recording = False
            self.record_button.set_label("Start Recording")
            if self.video_writer:
                self.video_writer.release()
                self.video_writer = None
                print(f"[INFO] Recording stopped and saved.")

if __name__ == "__main__":
    gui = DetectionGUI()
    Gtk.main()


ModuleNotFoundError: No module named 'onnxruntime'